## 11.7. The Transformer Architecture

In [1]:
import math
import pandas as pd
import torch
from torch import nn
from d2l import torch as d2l

### 11.7.1. Model

![](https://d2l.ai/_images/transformer.svg)

### 11.7.2. Positionwise Feed-Forward Networks

In [2]:
class PositionWiseFFN(nn.Module):  #@save
    """The positionwise feed-forward network."""
    def __init__(self, ffn_num_hiddens, ffn_num_outputs):
        super().__init__()
        self.dense1 = nn.LazyLinear(ffn_num_hiddens)
        self.relu = nn.ReLU()
        self.dense2 = nn.LazyLinear(ffn_num_outputs)

    def forward(self, X):
        return self.dense2(self.relu(self.dense1(X)))

In [10]:
ffn = PositionWiseFFN(4, 8)
ffn.eval()
X_test = torch.ones((2, 3, 4))
X_test.shape, X_test

(torch.Size([2, 3, 4]),
 tensor([[[1., 1., 1., 1.],
          [1., 1., 1., 1.],
          [1., 1., 1., 1.]],
 
         [[1., 1., 1., 1.],
          [1., 1., 1., 1.],
          [1., 1., 1., 1.]]]))

In [12]:
trans_1 = ffn.relu(ffn.dense1(X_test))
trans_1.shape, trans_1

(torch.Size([2, 3, 4]),
 tensor([[[0.0000, 0.1627, 0.0000, 0.0000],
          [0.0000, 0.1627, 0.0000, 0.0000],
          [0.0000, 0.1627, 0.0000, 0.0000]],
 
         [[0.0000, 0.1627, 0.0000, 0.0000],
          [0.0000, 0.1627, 0.0000, 0.0000],
          [0.0000, 0.1627, 0.0000, 0.0000]]], grad_fn=<ReluBackward0>))

In [15]:
y_ffn = ffn(X_test)
y_ffn.shape, y_ffn[0].shape, y_ffn[0]

(torch.Size([2, 3, 8]),
 torch.Size([3, 8]),
 tensor([[ 0.2574, -0.3693, -0.0925, -0.5465, -0.1285, -0.3039, -0.3186,  0.2045],
         [ 0.2574, -0.3693, -0.0925, -0.5465, -0.1285, -0.3039, -0.3186,  0.2045],
         [ 0.2574, -0.3693, -0.0925, -0.5465, -0.1285, -0.3039, -0.3186,  0.2045]],
        grad_fn=<SelectBackward0>))

### 11.7.3. Residual Connection and Layer Normalization

In [16]:
ln = nn.LayerNorm(2)
bn = nn.LazyBatchNorm1d()
X = torch.tensor([[1, 2], [2, 3]], dtype=torch.float32)
# Compute mean and variance from X in the training mode
print('layer norm:', ln(X), '\nbatch norm:', bn(X))

layer norm: tensor([[-1.0000,  1.0000],
        [-1.0000,  1.0000]], grad_fn=<NativeLayerNormBackward0>) 
batch norm: tensor([[-1.0000, -1.0000],
        [ 1.0000,  1.0000]], grad_fn=<NativeBatchNormBackward0>)


In [17]:
X.shape, X

(torch.Size([2, 2]),
 tensor([[1., 2.],
         [2., 3.]]))

In [37]:
m = torch.mean(X, dim=-1, keepdim=True)
v = torch.var(X, dim=-1, unbiased=False, keepdim=True)
m.shape, m, "", v.shape, v

(torch.Size([2, 1]),
 tensor([[1.5000],
         [2.5000]]),
 '',
 torch.Size([2, 1]),
 tensor([[0.2500],
         [0.2500]]))

In [38]:
(X - m) / torch.sqrt(v + eps)

tensor([[-1.0000,  1.0000],
        [-1.0000,  1.0000]])

In [39]:
class AddNorm(nn.Module):  #@save
    """The residual connection followed by layer normalization."""
    def __init__(self, norm_shape, dropout):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        self.ln = nn.LayerNorm(norm_shape)

    def forward(self, X, Y):
        return self.ln(self.dropout(Y) + X)

In [40]:
add_norm = AddNorm(4, 0.5)
shape = (2, 3, 4)
d2l.check_shape(add_norm(torch.ones(shape), torch.ones(shape)), shape)

### 11.7.4. Encoder

In [41]:
class TransformerEncoderBlock(nn.Module):  #@save
    """The Transformer encoder block."""
    def __init__(self, num_hiddens, ffn_num_hiddens, num_heads, dropout, use_bias=False):
        super().__init__()
        self.attention = self.attention = d2l.MultiHeadAttention(num_hiddens, num_heads, dropout, use_bias)
        self.addnorm1 = AddNorm(num_hiddens, dropout)
        self.ffn = PositionWiseFFN(ffn_num_hiddens, num_hiddens)
        self.addnorm2 = AddNorm(num_hiddens, dropout)

    def forward(self, X, valid_lens):
        Y = self.addnorm1(X, self.attention(X, X, X, valid_lens))
        return self.addnorm2(Y, self.ffn(Y))

In [42]:
X = torch.ones((2, 100, 24))
valid_lens = torch.tensor([3, 2])
encoder_blk = TransformerEncoderBlock(24, 48, 8, 0.5)
encoder_blk.eval()
d2l.check_shape(encoder_blk(X, valid_lens), X.shape)

In [44]:
X.shape, valid_lens.shape, valid_lens

(torch.Size([2, 100, 24]), torch.Size([2]), tensor([3, 2]))

In [45]:
torch.repeat_interleave(valid_lens, repeats=8, dim=0)

tensor([3, 3, 3, 3, 3, 3, 3, 3, 2, 2, 2, 2, 2, 2, 2, 2])

In [64]:
X_1 = X.reshape(X.shape[0], X.shape[1], 8, -1)
X_2 = X_1.permute(0, 2, 1, 3)
X_3 = X_2.reshape(-1, X_2.shape[2], X_2.shape[3])
X.shape, X_1.shape, X_2.shape, X_3.shape

(torch.Size([2, 100, 24]),
 torch.Size([2, 100, 8, 3]),
 torch.Size([2, 8, 100, 3]),
 torch.Size([16, 100, 3]))

In [92]:
d_test = X_3.shape[-1]
print(f"d_test: {d_test}\t|\t{X_3.transpose(1, 2).shape}\n")
bmm_1 = torch.bmm(X_3, X_3.transpose(1, 2)) / math.sqrt(d_test)
test_X_3 = torch.rand(16, 100, 3)
torch.all(test_X_3.transpose(1, 2) == test_X_3.permute(0, 2, 1))

d_test: 3	|	torch.Size([16, 3, 100])



tensor(True)

In [97]:
valid_lens2 = torch.repeat_interleave(valid_lens, repeats=8, dim=0)
valid_lens2.dim(), torch.repeat_interleave(valid_lens2, test_X_3.shape[1]).shape

(1, torch.Size([1600]))

In [99]:
class TransformerEncoder(d2l.Encoder):  #@save
    """The Transformer encoder."""
    def __init__(self, vocab_size, num_hiddens, ffn_num_hiddens, num_heads, num_blks, dropout, use_bias=False):
        super().__init__()
        self.num_hiddens = num_hiddens
        self.embedding = nn.Embedding(vocab_size, num_hiddens)
        self.pos_encoding = d2l.PositionalEncoding(num_hiddens, dropout)
        self.blks = nn.Sequential()
        for i in range(num_blks):
            self.blks.add_module("block"+str(i), TransformerEncoderBlock(
                num_hiddens, ffn_num_hiddens, num_heads, dropout, use_bias))

    def forward(self, X, valid_lens):
        # Since positional encoding values are between -1 and 1, the embedding
        # values are multiplied by the square root of the embedding dimension
        # to rescale before they are summed up
        X = self.pos_encoding(self.embedding(X) * math.sqrt(self.num_hiddens))
        self.attention_weights = [None] * len(self.blks)
        for i, blk in enumerate(self.blks):
            X = blk(X, valid_lens)
            self.attention_weights[i] = blk.attention.attention.attention_weights
        return X

In [101]:
encoder = TransformerEncoder(200, 24, 48, 8, 2, 0.5)
d2l.check_shape(encoder(torch.ones((2, 100), dtype=torch.long), valid_lens), (2, 100, 24))

### 11.7.5. Decoder